# Azure and GCP

## Overview

**Azure Machine Learning (Azure ML)** is Microsoft's managed ML platform. It covers the full MLOps lifecycle: experiment tracking, managed training, model registry, and real-time/batch endpoints. It integrates natively with **MLflow** for experiment tracking and **Azure DevOps / GitHub Actions** for CI/CD.

Azure ML is particularly common in organisations already using Microsoft infrastructure (Azure, Office 365, Active Directory) and is strong for hybrid cloud scenarios (cloud + on-premises).

## Core concepts

| Concept | Azure ML term | AWS SageMaker equivalent |
|---------|--------------|-------------------------|
| Project workspace | **Workspace** | Domain |
| Compute environment | **Compute cluster / instance** | Training instance |
| Experiment tracking | **MLflow / Experiments** | SageMaker Experiments |
| Training job | **Job** | Training job |
| Model storage | **Model Registry** | Model Registry |
| Serving endpoint | **Managed Online Endpoint** | SageMaker Endpoint |
| ML pipeline | **Azure ML Pipeline** | SageMaker Pipelines |
| AutoML | **AutoML** | SageMaker Autopilot |

## Installation

In [ ]:
# pip install azure-ai-ml azure-identity mlflow
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Authenticate and connect to a workspace
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id='<your-subscription-id>',
    resource_group_name='<your-resource-group>',
    workspace_name='<your-workspace>',
)

## Experiment tracking with MLflow

Azure ML uses MLflow as its native tracking backend — the same `mlflow` API you use locally, but runs are stored in the Azure ML workspace.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Set the MLflow tracking URI to your Azure ML workspace
# mlflow.set_tracking_uri(ml_client.workspaces.get(workspace_name).mlflow_tracking_uri)

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

with mlflow.start_run(run_name='rf-breast-cancer'):
    n_estimators = 100
    mlflow.log_param('n_estimators', n_estimators)

    model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metric('accuracy', acc)
    mlflow.sklearn.log_model(model, artifact_path='model')

    print(f'Accuracy: {acc:.4f}')

## Submitting a training job

In [ ]:
from azure.ai.ml import command, Input
from azure.ai.ml.entities import AmlCompute

# Define a managed compute cluster (auto-scales to 0 when idle)
compute = AmlCompute(
    name='cpu-cluster',
    size='Standard_DS3_v2',
    min_instances=0,
    max_instances=4,
)
# ml_client.begin_create_or_update(compute).result()

# Define and submit a command job
job = command(
    code='./src',                             # directory with your training script
    command='python train.py --lr ${{inputs.lr}}',
    inputs={'lr': Input(type='number', default=0.01)},
    environment='AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest',
    compute='cpu-cluster',
    display_name='sklearn-train-job',
)
# returned_job = ml_client.jobs.create_or_update(job)
# ml_client.jobs.stream(returned_job.name)  # stream logs

## Registering and deploying a model

In [ ]:
from azure.ai.ml.entities import Model, ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.ai.ml.constants import AssetTypes

# Register model from MLflow run
model = Model(
    path='azureml://jobs/<run-id>/outputs/artifacts/model',
    type=AssetTypes.MLFLOW_MODEL,
    name='breast-cancer-rf',
    description='Random forest on breast cancer dataset',
)
# registered = ml_client.models.create_or_update(model)

# Create a real-time endpoint
endpoint = ManagedOnlineEndpoint(name='breast-cancer-endpoint', auth_mode='key')
# ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# Deploy the model
deployment = ManagedOnlineDeployment(
    name='v1',
    endpoint_name='breast-cancer-endpoint',
    model='breast-cancer-rf:1',
    instance_type='Standard_DS3_v2',
    instance_count=1,
)
# ml_client.online_deployments.begin_create_or_update(deployment).result()

## Azure ML vs AWS SageMaker — key differences

| | Azure ML | AWS SageMaker |
|---|---|---|
| Experiment tracking | MLflow (native) | SageMaker Experiments (proprietary) |
| ML pipelines | Azure ML Pipelines | SageMaker Pipelines |
| AutoML | Azure AutoML | SageMaker Autopilot |
| Notebook environment | Compute Instances | SageMaker Studio / Notebooks |
| Model serving | Managed Online/Batch Endpoints | Real-time / Batch Transform |
| Integration strength | Azure ecosystem (Active Directory, DevOps) | AWS ecosystem (S3, Lambda, Step Functions) |
| Hybrid cloud | Strong (Azure Arc) | Limited |

## Further reading
- [Azure ML documentation](https://learn.microsoft.com/en-us/azure/machine-learning/)
- [Azure ML Python SDK v2](https://learn.microsoft.com/en-us/python/api/overview/azure/ai-ml-readme)
- [MLflow on Azure ML](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-use-mlflow-cli-runs)

---

## Overview

**Vertex AI** is Google Cloud's unified ML platform. It consolidates what was previously scattered across Cloud ML Engine, AutoML, and AI Platform into a single product. It has the tightest integration with **BigQuery** (for data at scale), **Google's foundation models** (Gemini, Imagen), and **Kubeflow Pipelines** for ML orchestration.

Vertex AI is the natural choice in GCP-native organisations and is particularly strong when your data already lives in BigQuery or GCS.

## Core concepts

| Concept | Vertex AI term | AWS SageMaker equivalent |
|---------|---------------|-------------------------|
| Project namespace | **Project + Region** | Domain |
| Compute for training | **Custom Training Job** | Training job |
| Experiment tracking | **Vertex AI Experiments** (MLflow-compatible) | SageMaker Experiments |
| Model storage | **Model Registry** | Model Registry |
| Serving endpoint | **Endpoint** | SageMaker Endpoint |
| ML pipeline | **Vertex AI Pipelines** (Kubeflow-based) | SageMaker Pipelines |
| AutoML | **AutoML** | SageMaker Autopilot |
| Notebook environment | **Workbench** | SageMaker Studio |
| Foundation models | **Model Garden** (Gemini, Imagen, Codey) | Bedrock (Claude, Llama, etc.) |

## Installation

In [ ]:
# pip install google-cloud-aiplatform
import vertexai
from google.cloud import aiplatform

# Initialise with your project and region
vertexai.init(project='your-gcp-project', location='us-central1')
aiplatform.init(project='your-gcp-project', location='us-central1')

## Submitting a custom training job

In [ ]:
from google.cloud import aiplatform

# Submit a custom training job using a pre-built container
job = aiplatform.CustomTrainingJob(
    display_name='sklearn-breast-cancer-rf',
    script_path='train.py',                         # your local training script
    container_uri='us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-0:latest',
    requirements=['scikit-learn', 'mlflow'],
    model_serving_container_image_uri=(
        'us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest'
    ),
)

# model = job.run(
#     machine_type='n1-standard-4',
#     replica_count=1,
# )

## Experiment tracking

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Vertex AI Experiments are MLflow-compatible
# mlflow.set_tracking_uri(aiplatform.get_experiment('my-experiment').resource_name)

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

with mlflow.start_run(run_name='rf-vertex-demo'):
    n_estimators = 100
    mlflow.log_param('n_estimators', n_estimators)

    model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metric('accuracy', acc)
    mlflow.sklearn.log_model(model, artifact_path='model')

    print(f'Accuracy: {acc:.4f}')

## Deploying a model to an endpoint

In [ ]:
from google.cloud import aiplatform

# Upload model to registry
# model = aiplatform.Model.upload(
#     display_name='breast-cancer-rf',
#     artifact_uri='gs://your-bucket/model/',    # GCS path
#     serving_container_image_uri=(
#         'us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest'
#     ),
# )

# Create endpoint and deploy
# endpoint = aiplatform.Endpoint.create(display_name='breast-cancer-endpoint')
# model.deploy(
#     endpoint=endpoint,
#     machine_type='n1-standard-2',
#     min_replica_count=1,
#     max_replica_count=3,   # auto-scales
# )

# Predict
# predictions = endpoint.predict(instances=X_test[:5].tolist())
# print(predictions.predictions)

## Vertex AI Pipelines (Kubeflow-based)

Vertex AI Pipelines use the **Kubeflow Pipelines (KFP) SDK** to define and orchestrate multi-step ML workflows.

In [ ]:
# pip install kfp google-cloud-pipeline-components
from kfp import dsl
from kfp.v2 import compiler

@dsl.component(base_image='python:3.11', packages_to_install=['scikit-learn'])
def train_model(n_estimators: int) -> float:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.datasets import load_breast_cancer
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))

@dsl.pipeline(name='breast-cancer-pipeline')
def my_pipeline(n_estimators: int = 100):
    train_task = train_model(n_estimators=n_estimators)

# compiler.Compiler().compile(my_pipeline, 'pipeline.json')
# job = aiplatform.PipelineJob(display_name='demo', template_path='pipeline.json')
# job.run()

## Using Google Foundation Models (Gemini)

Vertex AI's **Model Garden** provides access to Google's Gemini models for text, code, and multimodal tasks.

In [ ]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project='your-gcp-project', location='us-central1')

model = GenerativeModel('gemini-1.5-flash')  # or gemini-1.5-pro

response = model.generate_content(
    'Explain gradient boosting in two sentences.'
)
print(response.text)

## Vertex AI vs AWS SageMaker vs Azure ML

| | Vertex AI (GCP) | SageMaker (AWS) | Azure ML |
|---|---|---|---|
| Pipeline SDK | Kubeflow Pipelines (KFP) | Proprietary | Azure ML Pipelines |
| Experiment tracking | MLflow-compatible | SageMaker Experiments | MLflow (native) |
| Foundation models | Gemini, Imagen (Model Garden) | Claude, Llama (Bedrock) | OpenAI, Phi (Azure OpenAI) |
| AutoML | AutoML Tables/Vision/NLP | Autopilot | AutoML |
| Data integration | BigQuery-native | S3/Glue | Azure Data Lake / Synapse |
| Notebook environment | Workbench (managed JupyterLab) | Studio | Compute Instances |
| Hybrid/edge | Vertex AI Edge | SageMaker Edge | Azure Arc |

## Further reading
- [Vertex AI documentation](https://cloud.google.com/vertex-ai/docs)
- [Vertex AI Python SDK](https://cloud.google.com/python/docs/reference/aiplatform/latest)
- [Kubeflow Pipelines SDK](https://www.kubeflow.org/docs/components/pipelines/)
- [Gemini API in Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/start/quickstarts)